# Baseline

> Untuned small models scored with the code checks on the frozen test set

In [ ]:
#| default_exp baseline

The baseline scores small models before any fine-tuning, on the frozen test set and with the code checks, so that fine-tuning has a fair number to beat. Untuned models get a short prompt: the teacher's context and rules, and two worked examples. The teacher's bad message and failure-mode list would likely confuse a model of 1 to 2B parameters. The fine-tuned models get a one-line instruction instead, which training needs and the board favours, as [Fine-tuning](06_finetune.ipynb) explains.

In [ ]:
#| export
import json, hashlib
from collections import Counter
from IPython.display import Markdown
from mlx_lm import load, generate
from edgeslm.teacher import CONTEXT, RULES, EXAMPLES, user_prompt
from edgeslm.checks import check, parse
from edgeslm.dataset import read_jsonl, write_jsonl

In [ ]:
#| hide
from collections import Counter
from fastcore.test import *
from edgeslm.checks import parse

## The student prompt

A first baseline gave one example, with only the 🚨 and ⚠️ markers, and both models almost never used 👀, although most test items are early: small models copy the example's markers over the rules. The two examples here use all three markers, both kinds of problem, and both values of `soil_water`:

In [ ]:
#| export
STUDENT_EXAMPLES = [EXAMPLES[1], EXAMPLES[4]]

def student_prompt()->str:
    "The short system prompt for small models: context, rules and two worked examples"
    exs = '\n\n'.join(f"Record:\n{json.dumps(e['record'], ensure_ascii=False)}\n\nMessage:\n{e['message']}" for e in STUDENT_EXAMPLES)
    return f"{CONTEXT}\n\nRules:\n{RULES}\n\nExamples:\n\n{exs}\n\nReply with the message only."

def student_version(sp:str=None)->str:
    "A short hash of `sp`, the student prompt by default"
    return hashlib.sha256((sp or student_prompt()).encode()).hexdigest()[:12]

def student_input(
    tok, # The model's tokenizer
    rec:dict, # Triage record
    sp:str=None, # System prompt; the student prompt by default
)->str:
    "The chat-formatted prompt for `rec`, with thinking turned off"
    msgs = [dict(role='system', content=sp or student_prompt()), dict(role='user', content=user_prompt(rec))]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [ ]:
test_eq({m for e in STUDENT_EXAMPLES for m, _, _ in parse(e['message'])}, {'👀', '⚠️', '🚨'})
len(student_prompt()), student_version()

(4469, 'd46d8593fdef')

## Running a model

`run_model` writes a message for every row with greedy decoding, so that a run is reproducible, and checks each one:

In [ ]:
#| export
def run_model(
    model_id:str, # Hugging Face id or local path of an MLX model
    rows:list, # Rows with `id`, `tuple` and `record`
    adapter_path:str=None, # LoRA adapters to apply, for a fine-tuned model
    sp:str=None, # System prompt; the student prompt by default
    max_tokens:int=400, # Longest message to generate
)->list: # One row per input, with the message, its checks and the run's settings
    "Have `model_id` write and check a message for each row"
    model, tok = load(model_id, adapter_path=adapter_path)
    out = []
    for r in rows:
        msg = generate(model, tok, student_input(tok, r['record'], sp), max_tokens=max_tokens).strip()
        out.append(dict(id=r['id'], tuple=r['tuple'], message=msg, checks=check(r['record'], msg),
                        model=model_id, adapter=adapter_path, prompt=student_version(sp), decoding='greedy'))
    return out

This cell wrote the baseline runs of Qwen3.5 0.8B and 2B, in full precision. Both together took about 18 minutes on an Apple M5, and the cell does not run in tests:

In [ ]:
#| eval: false
test = read_jsonl('../data/test-v1.jsonl')
for m, f in [('mlx-community/Qwen3.5-0.8B-bf16', 'qwen3.5-0.8b'), ('mlx-community/Qwen3.5-2B-bf16', 'qwen3.5-2b')]:
    write_jsonl(f'../data/baseline-v2-{f}.jsonl', run_model(m, test))

## Results

`score` counts the messages that pass each check, and all of them. `score_table` puts several runs side by side:

In [ ]:
#| export
def score(rows:list)->dict:
    "Messages passing all checks, then each check"
    return dict(all=sum(all(r['checks'].values()) for r in rows), **{k: sum(r['checks'][k] for r in rows) for k in rows[0]['checks']})

def score_table(
    runs:dict, # Run name -> its rows
)->Markdown:
    "A Markdown table of `score` for each run, out of the number of rows"
    ss = {k: score(v) for k, v in runs.items()}
    head = '| Check | ' + ' | '.join(runs) + ' |\n|---|' + '---|'*len(runs) + '\n'
    return Markdown(head + '\n'.join(f'| {c} | ' + ' | '.join(f"{s[c]} / {len(runs[n])}" for n, s in ss.items()) + ' |' for c in next(iter(ss.values()))))

Both baselines, with the first one-example run for comparison:

In [ ]:
runs = {f'{m} {v}': read_jsonl(f'../data/baseline-{v}-qwen3.5-{m}.jsonl') for v in ('v1', 'v2') for m in ('0.8b', '2b')}
score_table(runs)

| Check | 0.8b v1 | 2b v1 | 0.8b v2 | 2b v2 |
|---|---|---|---|---|
| all | 31 / 270 | 64 / 270 | 123 / 270 | 70 / 270 |
| format | 270 / 270 | 270 / 270 | 269 / 270 | 270 / 270 |
| coverage | 211 / 270 | 269 / 270 | 269 / 270 | 269 / 270 |
| no_invented | 191 / 270 | 268 / 270 | 270 / 270 | 270 / 270 |
| severity | 78 / 270 | 73 / 270 | 143 / 270 | 79 / 270 |
| no_numbers | 237 / 270 | 255 / 270 | 269 / 270 | 264 / 270 |
| length | 267 / 270 | 266 / 270 | 267 / 270 | 266 / 270 |
| amount | 215 / 270 | 213 / 270 | 234 / 270 | 237 / 270 |

`markers` pairs the true severity of each item with the marker the model wrote for it:

In [ ]:
#| export
def markers(
    rows:list, # Scored rows of one run
    test:list, # The test records, which hold each item's true severity
)->Counter:
    "Count each (true severity, written marker) pair over the items of `rows`"
    sev = {(r['id'], it['location'].casefold()): it['severity'] for r in test for it in r['record']['items']}
    return Counter((sev.get((r['id'], l.casefold())), m) for r in rows for m, l, _ in filter(None, parse(r['message'])))

The second prompt more than doubled the 0.8B's score, but neither model maps the three severities to the three markers. The 0.8B writes 👀 for nearly every early or moderate item and 🚨 for severe ones. The 2B writes 🚨 for most items, whatever their severity:

In [ ]:
test = read_jsonl('../data/test-v1.jsonl')
markers(runs['0.8b v2'], test), markers(runs['2b v2'], test)

(Counter({('early', '👀'): 191,
          ('moderate', '👀'): 122,
          ('severe', '🚨'): 103,
          ('moderate', '🚨'): 25,
          ('early', '🚨'): 17,
          ('severe', '👀'): 9,
          ('early', '⚠️'): 1}),
 Counter({('early', '🚨'): 159,
          ('severe', '🚨'): 112,
          ('moderate', '🚨'): 105,
          ('early', '👀'): 34,
          ('moderate', '⚠️'): 27,
          ('early', '⚠️'): 16,
          ('moderate', '👀'): 15}))

The test items are 45% early, so a model that never writes ⚠️ still gets most markers right, and that is where the 0.8B's lead over the 2B comes from:

In [ ]:
Counter(it['severity'] for r in test for it in r['record']['items'])

Counter({'early': 209, 'moderate': 148, 'severe': 112})

The baseline depends heavily on the prompt's examples. That is itself a reason to fine-tune: a fine-tuned model should learn the mapping from many examples, not copy the markers of two.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()